# netstead quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/e-lo/netstead/blob/refactor/v1.0/packages/netstead/docs/notebooks/quickstart.ipynb)

Five cells, end-to-end: install, load a GMNS network straight from a public URL,
validate it, render the new map+table HTML viewer inline, and drill into a single
link via ibis.

Run in [Colab](https://colab.research.google.com/) or any Jupyter environment.
No local files needed — the Leavenworth fixture is loaded over HTTPS from this repo.

In [ ]:
# 1. Install netstead with the optional extras we need:
#    - [osm] for build_network_from_osm (not used in this notebook, but cheap)
#    - [graph] for scope.from_nodes / connectivity (not used here, but cheap)
#    - [reports] for the network-on-map HTML viewer
%pip install -q 'netstead[osm,graph,reports]'

In [ ]:
# 2. Load a GMNS network from a raw GitHub URL — no clone, no local files.
from netstead import Network

FIXTURE = "https://raw.githubusercontent.com/e-lo/netstead/refs/heads/main/packages/netstead/netstead/fixtures/leavenworth/csv"
net = Network.from_source(FIXTURE)
net  # _repr_html_ shows GMNS spec + link/node count

In [ ]:
# 3. Validate against the embedded GMNS spec (the four-pass validator:
#    structural / schema / foreign-key / sync-state).
report = net.validate()
report  # _repr_html_ shows per-severity counts + first 10 issues

In [ ]:
# 4. Render the interactive map+table viewer — open inline in the notebook.
from IPython.display import IFrame
from netstead.reports import render_validation_html

out_path = "/tmp/leavenworth_report.html"
with open(out_path, "w") as f:
    f.write(render_validation_html(net, report))
IFrame(out_path, width=900, height=600)

In [ ]:
# 5. Drill into a single link via ibis (no need to materialise the whole table).

first_link_id = int(net.links.to_pandas().iloc[0]["link_id"])
net.links.filter(lambda t: t.link_id == first_link_id).to_pandas()